In [ ]:
from google.colab import files

uploaded = files.upload()

Saving qa_data.csv to qa_data.csv


In [ ]:
import pandas as pd

# Load the original dataset
df = pd.read_csv("qa_data.csv")

# Display dataset information
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# Display first 5 rows
display(df.head())

Dataset shape: (10000, 4)

Columns: ['knowledge', 'question', 'right_answer', 'hallucinated_answer']


,knowledge,question,right_answer,hallucinated_answer
0,Arthur's Magazine (1844–1846) was an American ...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,First for Women was started first.
1,The Oberoi family is an Indian family that is ...,The Oberoi family is part of a hotel company t...,Delhi,The Oberoi family's hotel company is based in ...
2,"Allison Beth ""Allie"" Goertz (born March 2, 199...",Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,"Allie Goertz wrote a song about Milhouse, a po..."
3,"Margaret ""Peggy"" Seeger (born June 17, 1935) i...",What nationality was James Henry Miller's wife?,American,James Henry Miller's wife was British.
4,It is a hygroscopic solid that is highly solu...,Cadmium Chloride is slightly soluble in this c...,alcohol,water with a hint of alcohol


In [ ]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicate records
print("\nDuplicate records:", df.duplicated().sum())

# Check data types
print("\nData types:")
print(df.dtypes)

Missing values:
knowledge              0
question               0
right_answer           0
hallucinated_answer    0
dtype: int64

Duplicate records: 0

Data types:
knowledge              object
question               object
right_answer           object
hallucinated_answer    object
dtype: object


In [ ]:
# Create correct answer examples
correct = pd.DataFrame({
    "question": df["question"],
    "answer": df["right_answer"],
    "label": 0
})

# Create hallucinated answer examples
hallucinated = pd.DataFrame({
    "question": df["question"],
    "answer": df["hallucinated_answer"],
    "label": 1
})

# Combine both datasets
training_df = pd.concat(
    [correct, hallucinated],
    ignore_index=True
)

# Shuffle the dataset
training_df = training_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Save the prepared dataset
training_df.to_csv("training_data.csv", index=False)

print("Training dataset created!")
print("Shape:", training_df.shape)
display(training_df.head())

Training dataset created!
Shape: (20000, 3)


,question,answer,label
0,Which 2004 Nickelodeon show was created by Sue...,"The animated series ""Pepper Ann"" and ""Angela A...",1
1,Non-Stop starred the English actress best know...,Lady Mary Crawley,0
2,Cross country running is a sport in which team...,dirt or grass,0
3,King of Queens is the debut studio album by Ni...,R2Bees,0
4,What is the nationality of the writer and dire...,The writer and director of The Tree of Life we...,1


In [ ]:
print("Dataset shape:", training_df.shape)

print("\nLabel distribution:")
print(training_df["label"].value_counts())

print("\nLabel percentages:")
print(training_df["label"].value_counts(normalize=True) * 100)

Dataset shape: (20000, 3)

Label distribution:
label
1    10000
0    10000
Name: count, dtype: int64

Label percentages:
label
1    50.0
0    50.0
Name: proportion, dtype: float64


In [ ]:
# Display 5 questions and answers
display(training_df[["question", "answer"]].head())

,question,answer
0,Which 2004 Nickelodeon show was created by Sue...,"The animated series ""Pepper Ann"" and ""Angela A..."
1,Non-Stop starred the English actress best know...,Lady Mary Crawley
2,Cross country running is a sport in which team...,dirt or grass
3,King of Queens is the debut studio album by Ni...,R2Bees
4,What is the nationality of the writer and dire...,The writer and director of The Tree of Life we...


In [ ]:
import re

def clean_text(text):
    # Convert text to string
    text = str(text)

    # Remove leading and trailing spaces
    text = text.strip()

    # Replace multiple spaces, tabs, and newlines
    # with a single space
    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
# Clean questions
training_df["question"] = training_df["question"].apply(clean_text)

# Clean answers
training_df["answer"] = training_df["answer"].apply(clean_text)

print("Text cleaning completed!")

Text cleaning completed!


In [ ]:
# Count empty questions and answers
print("Empty questions:",
      (training_df["question"] == "").sum())

print("Empty answers:",
      (training_df["answer"] == "").sum())

Empty questions: 0
Empty answers: 0


In [ ]:
# Display cleaned examples
display(training_df.head())

# Check for missing values
print("Missing values:")
print(training_df.isnull().sum())

# Check data types
print("\nData types:")
print(training_df.dtypes)

,question,answer,label
0,Which 2004 Nickelodeon show was created by Sue...,"The animated series ""Pepper Ann"" and ""Angela A...",1
1,Non-Stop starred the English actress best know...,Lady Mary Crawley,0
2,Cross country running is a sport in which team...,dirt or grass,0
3,King of Queens is the debut studio album by Ni...,R2Bees,0
4,What is the nationality of the writer and dire...,The writer and director of The Tree of Life we...,1


Missing values:
question    0
answer      0
label       0
dtype: int64

Data types:
question    object
answer      object
label        int64
dtype: object


In [ ]:
training_df.to_csv(
    "cleaned_halueval.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [ ]:
from google.colab import files

files.download("cleaned_halueval.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================================================
# REMAINING DATA PREPROCESSING
# Starting after text cleaning
# =====================================================

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# STEP 1: Validate the cleaned dataset

df = training_df.copy()

required_columns = ["question", "answer", "label"]

assert all(col in df.columns for col in required_columns), \
    "Required columns are missing!"

assert df[required_columns].notnull().all().all(), \
    "Missing values found!"

assert df["label"].isin([0, 1]).all(), \
    "Invalid labels found!"

print("Dataset validation successful!")
print("Dataset shape:", df.shape)


# STEP 2: Remove duplicate examples

before = len(df)

df = df.drop_duplicates(
    subset=["question", "answer", "label"]
).copy()

print("\nDuplicate examples removed:", before - len(df))
print("Dataset shape after duplicate removal:", df.shape)


# STEP 3: Split by question to prevent data leakage

# Keep all answers belonging to the same question
# in the same dataset.

groups = df["question"]

# First, separate 15% for testing
split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

train_val_idx, test_idx = next(
    split1.split(df, groups=groups)
)

train_val_df = df.iloc[train_val_idx].copy()
test_df = df.iloc[test_idx].copy()


# Next, separate 15% of the total for validation
split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15 / 0.85,
    random_state=42
)

train_idx, val_idx = next(
    split2.split(
        train_val_df,
        groups=train_val_df["question"]
    )
)

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()


# STEP 4: Reset indexes

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


# STEP 5: Check for data leakage

train_questions = set(train_df["question"])
val_questions = set(val_df["question"])
test_questions = set(test_df["question"])

print("\nData leakage check:")
print("Train-validation overlap:",
      len(train_questions & val_questions))

print("Train-test overlap:",
      len(train_questions & test_questions))

print("Validation-test overlap:",
      len(val_questions & test_questions))


# STEP 6: Check label distribution

print("\nLabel distribution:")

for name, data in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(f"\n{name} set:")
    print(data["label"].value_counts().sort_index())
    print("Percentages:")
    print(
        (data["label"].value_counts(normalize=True)
         .sort_index() * 100).round(2)
    )


# STEP 7: Save the datasets as CSV files

df.to_csv("preprocessed_halueval.csv", index=False)

train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)


# STEP 8: Final summary

print("\n" + "=" * 45)
print("PREPROCESSING COMPLETED")
print("=" * 45)

print("Complete dataset:", df.shape)
print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Testing set:", test_df.shape)

print("\nFiles saved:")
print("1. preprocessed_halueval.csv")
print("2. train.csv")
print("3. validation.csv")
print("4. test.csv")

print("\nSample training data:")
display(train_df.head())

Dataset validation successful!
Dataset shape: (20000, 3)

Duplicate examples removed: 0
Dataset shape after duplicate removal: (20000, 3)

Data leakage check:
Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0

Label distribution:

Training set:
label
0    7000
1    7000
Name: count, dtype: int64
Percentages:
label
0    50.0
1    50.0
Name: proportion, dtype: float64

Validation set:
label
0    1500
1    1500
Name: count, dtype: int64
Percentages:
label
0    50.0
1    50.0
Name: proportion, dtype: float64

Testing set:
label
0    1500
1    1500
Name: count, dtype: int64
Percentages:
label
0    50.0
1    50.0
Name: proportion, dtype: float64

PREPROCESSING COMPLETED
Complete dataset: (20000, 3)
Training set: (14000, 3)
Validation set: (3000, 3)
Testing set: (3000, 3)

Files saved:
1. preprocessed_halueval.csv
2. train.csv
3. validation.csv
4. test.csv

Sample training data:


,question,answer,label
0,Which 2004 Nickelodeon show was created by Sue...,"The animated series ""Pepper Ann"" and ""Angela A...",1
1,Non-Stop starred the English actress best know...,Lady Mary Crawley,0
2,Cross country running is a sport in which team...,dirt or grass,0
3,King of Queens is the debut studio album by Ni...,R2Bees,0
4,What county is the Moscow Hemp Fest held in?,The Moscow Hemp Fest is held in Idaho County.,1
